## Data Processing

In [15]:
import numpy as np
import pandas as pd
import seaborn as sns
import datetime as dt
import matplotlib.pyplot as plt
import yfinance as yf
import torch.nn as nn

In [40]:
# asset price loading
# dropping Null vals
# , 'TLT', 'XLF', 'XLE', 'XLU', 'XLK', 'EXW1.DE', '^FTSE','1329.T', 'QQQ'
symbols = yf.download(['SPY'],start=dt.datetime(2009,1,1), end=dt.datetime(2016,12,31), auto_adjust=False, multi_level_index=False, progress=False).dropna()

closePs = symbols['Adj Close']
closePs

Date
2009-01-02     67.806602
2009-01-05     67.726357
2009-01-06     68.178581
2009-01-07     66.136223
2009-01-08     66.406105
                 ...    
2016-12-23    194.372925
2016-12-27    194.855103
2016-12-28    193.244766
2016-12-29    193.201736
2016-12-30    192.495529
Name: Adj Close, Length: 2014, dtype: float64

In [41]:
import torch
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

cuda:0


In [ ]:
prices = closePs.to_numpy()
dates = closePs.index

## Optimizer from scratch

In [ ]:
# Sharpness aware minimization based optim, it's better than SDG w/ momentum

class myOptimizer():

    def __init__(self):

    def _get_batch(self):

    def _get_momentum_vector(self, X_batch, y_batch):

    def fit(self, X, y, batch_size = 32, epochs = 100):


    def predict(self, X):
	    return self.w * X + self.b

In [ ]:
#Will train models using SAM
from tqdm import tqdm
from src.sam import SAM

def train_epoch_sam(
    model,
    loader,
    criterion,
    optimizer: SAM,
    device,
    use_amp: bool,
):

    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for x, y in tqdm(loader, desc="train", leave=False):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        if use_amp:
            with amp.autocast("cuda"):
                logits = model(x)
                loss = criterion(logits, y)
            loss.backward()
            optimizer.first_step(zero_grad=True)

            with amp.autocast("cuda"):
                logits2 = model(x)
                loss2 = criterion(logits2, y)
            loss2.backward()
            optimizer.second_step(zero_grad=True)
        else:
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.first_step(zero_grad=True)

            logits2 = model(x)
            loss2 = criterion(logits2, y)
            loss2.backward()
            optimizer.second_step(zero_grad=True)

        total_loss += loss.detach().float().item() * y.size(0)
        correct += (logits2.argmax(dim=1) == y).sum().item()
        total += y.size(0)
    return total_loss / max(total, 1), correct / max(total, 1)